<div align="center">
  <h1 style="font-size: 40px; background: linear-gradient(135deg, #7c3aed, #db2777);
      -webkit-background-clip: text; -webkit-text-fill-color: transparent;
      font-family: sans-serif; margin-bottom: 4px;">⚡ VideoClipse</h1>
  <h3 style="color: #94a3b8; font-family: sans-serif; margin-top: 0;">AI Video Studio — Colab Edition</h3>
  <p style="color: #64748b; font-size: 13px;">
    Jalanin cell 1 -> 2 -> 3 -> dapet link aplikasi!
  </p>
  <hr style="border: 1px solid #1e293b; margin: 16px 0;">
</div>

In [ ]:
# ============================================================
# 1. INSTALL SEMUA DEPENDENCIES
# ============================================================
import subprocess, sys, os, time, threading, json, urllib.request

print("Install ffmpeg...")
subprocess.run(["apt-get", "update", "-qq"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], capture_output=True)

print("Install Python packages...")
pkgs = [
    "yt-dlp>=2024.12.0",
    "faster-whisper>=1.1.0",
    "streamlit>=1.35.0",
    "requests>=2.31.0",
    "Pillow>=10.0.0",
    "pyngrok>=7.0.0",
    "playwright>=1.48.0",
]
for pkg in pkgs:
    print(f"   {pkg}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"      ERROR: {r.stderr[-100:]}")

print("\nVerifikasi:")
r = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print(f"   {r.stdout.split(chr(10))[0]}")
r = subprocess.run([sys.executable, "-m", "streamlit", "--version"],
                   capture_output=True, text=True)
print(f"   Streamlit {r.stdout.strip()}")
print("\nSelesai! Lanjut ke cell 2.")

In [ ]:
# ============================================================
# 2. MOUNT GOOGLE DRIVE + DOWNLOAD CODE
# ============================================================
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/VideoClipse_Output'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Output folder: {DRIVE_DIR}")

# Clone repo
REPO = "https://github.com/nenocahya22-lgtm/cliper.git"
APP_DIR = "/content/VideoClipse"

if not Path(APP_DIR).exists():
    print("Download VideoClipse dari GitHub...")
    subprocess.run(["git", "clone", "--depth", "1", REPO, APP_DIR],
                   capture_output=True, text=True)

os.chdir(APP_DIR)
print(f"App directory: {APP_DIR}")
print("\nSelesai! Lanjut ke cell 3.")

In [ ]:
# ============================================================
# 3. START STREAMLIT APP + NGORK TUNNEL
# ============================================================

# Fallback: install pyngrok kalo gagal di cell 1
try:
    from pyngrok import ngrok
except ImportError:
    print("pyngrok belum terinstall. Install dulu...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok
    print("pyngrok terinstall!")

NGROK_AUTH = ""  # <-- ISI TOKEN NGROK LO DI SINI (GRATIS)

if NGROK_AUTH:
    ngrok.set_auth_token(NGROK_AUTH)
    print("Token ngrok terpasang!")
else:
    print("Kalo error, daftar gratis di https://ngrok.com lalu paste token di atas")

# Kill port 8501 kalo ada yg pake
subprocess.run(["fuser", "-k", "8501/tcp"], capture_output=True)
time.sleep(1)

# Start Streamlit di background
print("\nStart Streamlit app...")
os.environ["STREAMLIT_SERVER_PORT"] = "8501"
os.environ["STREAMLIT_SERVER_HEADLESS"] = "true"
os.environ["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"

subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(8)

# Buka tunnel ngrok
print("Buka tunnel publik...")
try:
    tunnel = ngrok.connect(8501, proto="http")
    URL = tunnel.public_url
except Exception as e:
    URL = "http://localhost:8501"
    print(f"Ngrok error: {e}")

print("\n" + "=" * 60)
print("  APLIKASI SIAP!")
print("=" * 60)
print(f"\n  BUKA LINK INI DI BROWSER:")
print(f"  {URL}")
print(f"\n  (CTRL+click atau buka manual)")
print("=" * 60)
print("\nApp berjalan di background Colab.")
print("Jangan tutup tab Colab ini selama pake app.")
print("\nMau stop? Klik Runtime > Interrupt execution.")

---

### Cara pake
1. Click link di atas
2. Login pake username & password (daftar kalo belum)
3. Paste link YouTube, klik **New Clip**
4. Pilih momen, edit, render, selesai!

### Hasil video
Tersimpan otomatis di Google Drive:
`MyDrive > VideoClipse_Output`

### Catatan
- Colab gratis: 12 jam session, 90 menit idle
- Video output aman di Google Drive walau session mati
- Kalo mau pake lagi, tinggal buka link Colab lagi & run ulang

---
<div align="center">
  <p style="color: #64748b;">
    ⚡ VideoClipse — AI Video Studio<br>
    Dibuat dengan ❤️ buat content creator Indonesia
  </p>
</div>